In [2]:
import torch
from usta_model import UstaModel
from usta_tokenizer import UstaTokenizer

u_tokenizer = UstaTokenizer("tokenizer.json")

prompt = "the capital of the united"

tokens = u_tokenizer.encode(prompt)
tokens

tensor([ 0, 61,  1, 61,  2, 61,  0, 61,  3])

In [5]:
context_length = 32

In [6]:
torch.manual_seed(1)
u_model = UstaModel(vocab_size=len(u_tokenizer.vocab), embedding_dim=12, num_heads=4, context_length=context_length, num_layers=8)

out = u_model(tokens)
out.shape

torch.Size([9, 64])

In [7]:
with open("text.txt", "r") as f:
  text = f.read()

len(text), text[:100]

(4101,
 'the capital of the united states is not london. the capital of france is paris, and berlin is the ca')

In [8]:
token_ids = u_tokenizer.encode(text)
len(token_ids), type(token_ids)

(1593, torch.Tensor)

In [9]:
ids = token_ids.detach().cpu().numpy().tolist()
len(ids), type(ids)

(1593, list)

In [10]:
from text_dataset import TextDataset

stride = 12

dataset = TextDataset(ids, context_length, stride)

len(dataset.inputs), len(dataset.targets)

(131, 131)

In [11]:
dataset.inputs[0], dataset.targets[0]

(tensor([ 0, 61,  1, 61,  2, 61,  0, 61,  3, 61,  4, 58, 61,  5, 61,  6, 61,  7,
         59, 61,  0, 61,  1, 61,  2, 61,  8, 61,  5, 61,  9, 60]),
 tensor([61,  1, 61,  2, 61,  0, 61,  3, 61,  4, 58, 61,  5, 61,  6, 61,  7, 59,
         61,  0, 61,  1, 61,  2, 61,  8, 61,  5, 61,  9, 60, 61]))

In [ ]:
# model parameters count
parameters_count = sum(p.numel() for p in u_model.parameters())
print(parameters_count)

# model architecture
print(u_model)

11776
UstaModel(
  (embedding): UstaEmbedding(
    (embedding): Embedding(64, 12)
  )
  (layers): Sequential(
    (0): UstaDecoderBlock(
      (self_attention): UstaMultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=12, out_features=12, bias=True)
        )
        (projection): Linear(in_features=12, out_features=12, bias=True)
      )
      (norm1): UstaLayerNorm()
      (mlp): UstaMLP(
        (gate_proj): Linear(in_features=12, out_features=12, bias=True)
        (up_proj): Linear(in_features=12, out_features=12, bias=True)
        (down_proj): Linear(in_features=12, out_features=12, bias=True)
        (gelu): GELU()
      )
      (norm2): UstaLayerNorm()
    )
    (1): UstaDecoderBlock(
      (self_attention): UstaMultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=12, out_features=12, bias=True)
        )
    

In [16]:
out0 = u_model(dataset.inputs[0])
out0.shape

torch.Size([32, 64])

In [17]:
import torch.nn as nn

loss_fn = nn.CrossEntropyLoss()

In [18]:
loss = loss_fn(out0, dataset.targets[0])
loss

tensor(4.3883, grad_fn=<NllLossBackward0>)

In [19]:
loss.item()

4.388274669647217

In [20]:
# optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)
optimizer = torch.optim.AdamW(u_model.parameters(), lr=1e-3)


In [21]:
for input, target in dataset:
  print(input.shape, target.shape)
  break

torch.Size([32]) torch.Size([32])


In [72]:
epoch = 9000

for epoch in range(epoch):
  total_loss = 0.
  for input, target in dataset:
    pred = u_model(input)
    
    loss = loss_fn(pred, target)
    total_loss += loss.item()
    
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

  average_loss = total_loss / len(dataset)
  print(f"Epoch {epoch + 1} loss: {loss.item()} average loss: {average_loss}")
    

Epoch 1 loss: 0.5149598121643066 average loss: 0.5281891147143968
Epoch 2 loss: 0.40350404381752014 average loss: 0.5175691062712487
Epoch 3 loss: 0.7075916528701782 average loss: 0.5301317548706331
Epoch 4 loss: 0.6390193700790405 average loss: 0.5308646716689336
Epoch 5 loss: 0.6248183250427246 average loss: 0.5515484560764473
Epoch 6 loss: 0.5792657732963562 average loss: 0.5829925125336829
Epoch 7 loss: 0.47671741247177124 average loss: 0.5610494184812517
Epoch 8 loss: 0.6278808116912842 average loss: 0.5281715978875415
Epoch 9 loss: 0.619047999382019 average loss: 0.536191534108788
Epoch 10 loss: 0.761023223400116 average loss: 0.49814695301856704
Epoch 11 loss: 0.4357930123806 average loss: 0.5140318995668688
Epoch 12 loss: 0.43218958377838135 average loss: 0.5122819094712497
Epoch 13 loss: 0.6410170793533325 average loss: 0.522387859930519
Epoch 14 loss: 0.5772663950920105 average loss: 0.5179152273722277
Epoch 15 loss: 1.0407168865203857 average loss: 0.5167254545078933
Epoch 1

In [22]:
new_tokens = tokens.detach().cpu().numpy().tolist()
new_tokens.append(61)
new_tokens


[0, 61, 1, 61, 2, 61, 0, 61, 3, 61]

In [23]:
import torch

new_tokens = u_tokenizer.encode("madrid is in")
new_tokens = new_tokens.detach().cpu().numpy().tolist()
new_tokens.append(61)

out = u_model(torch.tensor(new_tokens))

probs = torch.softmax(out[-1], dim=-1)
max_prob, max_index = torch.max(probs, dim=-1)
max_prob, max_index, probs

(tensor(0.0859, grad_fn=<MaxBackward0>),
 tensor(60),
 tensor([0.0122, 0.0067, 0.0045, 0.0453, 0.0040, 0.0298, 0.0124, 0.0037, 0.0059,
         0.0118, 0.0171, 0.0170, 0.0207, 0.0024, 0.0034, 0.0129, 0.0057, 0.0037,
         0.0069, 0.0382, 0.0428, 0.0108, 0.0076, 0.0049, 0.0032, 0.0357, 0.0124,
         0.0029, 0.0137, 0.0202, 0.0031, 0.0271, 0.0023, 0.0043, 0.0047, 0.0132,
         0.0240, 0.0035, 0.0163, 0.0448, 0.0069, 0.0210, 0.0506, 0.0279, 0.0092,
         0.0084, 0.0290, 0.0070, 0.0054, 0.0052, 0.0243, 0.0226, 0.0077, 0.0069,
         0.0284, 0.0260, 0.0137, 0.0121, 0.0041, 0.0051, 0.0859, 0.0085, 0.0079,
         0.0146], grad_fn=<SoftmaxBackward0>))

In [24]:
# save model
torch.save(u_model.state_dict(), "u_model.pth")

# load model
u_model.load_state_dict(torch.load("u_model.pth"))

# generate text
new_tokens = u_tokenizer.encode("the capital of the united states is london. the capital of france is")
new_tokens = new_tokens.detach().cpu().numpy().tolist()
new_tokens.append(61)
len(new_tokens)

28

In [25]:
loaded_model = UstaModel(64, embedding_dim=12, num_heads=4, context_length=32, num_layers=8)
loaded_model.load_state_dict(torch.load("u_model.pth"))
loaded_model

UstaModel(
  (embedding): UstaEmbedding(
    (embedding): Embedding(64, 12)
  )
  (layers): Sequential(
    (0): UstaDecoderBlock(
      (self_attention): UstaMultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=12, out_features=12, bias=True)
        )
        (projection): Linear(in_features=12, out_features=12, bias=True)
      )
      (norm1): UstaLayerNorm()
      (mlp): UstaMLP(
        (gate_proj): Linear(in_features=12, out_features=12, bias=True)
        (up_proj): Linear(in_features=12, out_features=12, bias=True)
        (down_proj): Linear(in_features=12, out_features=12, bias=True)
        (gelu): GELU()
      )
      (norm2): UstaLayerNorm()
    )
    (1): UstaDecoderBlock(
      (self_attention): UstaMultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=12, out_features=12, bias=True)
        )
        (p

In [26]:
out = u_model(torch.tensor(new_tokens))

probs = torch.softmax(out[-1], dim=-1)
max_prob, max_index = torch.max(probs, dim=-1)
max_prob, max_index, probs

(tensor(0.0816, grad_fn=<MaxBackward0>),
 tensor(60),
 tensor([0.0126, 0.0088, 0.0031, 0.0429, 0.0023, 0.0478, 0.0129, 0.0038, 0.0086,
         0.0072, 0.0143, 0.0129, 0.0243, 0.0024, 0.0031, 0.0155, 0.0057, 0.0058,
         0.0067, 0.0514, 0.0511, 0.0109, 0.0101, 0.0029, 0.0030, 0.0222, 0.0121,
         0.0033, 0.0191, 0.0177, 0.0027, 0.0200, 0.0042, 0.0061, 0.0037, 0.0106,
         0.0427, 0.0055, 0.0156, 0.0396, 0.0074, 0.0240, 0.0468, 0.0181, 0.0117,
         0.0064, 0.0213, 0.0048, 0.0038, 0.0028, 0.0221, 0.0225, 0.0083, 0.0104,
         0.0298, 0.0232, 0.0103, 0.0082, 0.0025, 0.0050, 0.0816, 0.0111, 0.0053,
         0.0174], grad_fn=<SoftmaxBackward0>))